# 🤖 AI, CA3, Machine Learning 📚  

* **Name** : [Enter your name] 🖊  
* **Last Name** : [Enter your last name] 📝  
* **SID** : [Enter your SID] 🆔

## Part 1


### 🛠 From Scratch  
In this subsection, you should implement the Decision Tree algorithm 🌳 from scratch 🛠 and compare its performance with the library implementation 📚.

In [2]:
import numpy as np
import pandas as pd

class MyDecisionTree:
    def __init__(self, max_depth=3, min_samples_split=2, min_samples_leaf=1):
        """
        max_depth : int
            Maximum depth of the tree (root has depth 0).
        """
        self.max_depth = int(max_depth)
        self.min_samples_split = int(min_samples_split)
        self.min_samples_leaf = int(min_samples_leaf)
        self.root = None

    def fit(self, X, y):
        """
        Build the decision tree from scratch.

        X : pd.DataFrame (n_samples, n_features)
        y : pd.Series (n_samples,)
        """
        X = self._to_df(X)
        y = self._to_series(y)

        self.features_ = list(X.columns)
        self.root = self._build_tree(X, y, depth=0)
        return self

    def predict(self, X):
        """
        Predict class labels for samples in X.

        Returns
        -------
        preds : np.ndarray (n_samples,)
        """
        X = self._to_df(X)
        preds = np.zeros(len(X), dtype=int)
        for i, (_, row) in enumerate(X.iterrows()):
            preds[i] = self._predict_one(row, self.root)
        return preds

    def _to_df(self, X):
        if isinstance(X, pd.DataFrame):
            return X
        return pd.DataFrame(X)

    def _to_series(self, y):
        if isinstance(y, pd.Series):
            return y.reset_index(drop=True)
        return pd.Series(y)

    def _gini(self, y):
        if len(y) == 0:
            return 0.0
        p = y.value_counts(normalize=True).values
        return 1.0 - np.sum(p ** 2)

    def _majority_class(self, y):
        vc = y.value_counts()
        mx = vc.max()
        winners = vc[vc == mx].index.tolist()
        return int(min(winners))

    def _best_split(self, X, y):
        """
        Find best (feature, threshold) to split on.
        For numeric features: split by <= threshold vs > threshold.
        For categorical features: split by == category vs != category.
        Returns dict or None.
        """
        best = None
        best_impurity = float("inf")
        parent_gini = self._gini(y)
        n = len(y)
        if n < self.min_samples_split or parent_gini == 0.0:
            return None

        for feat in X.columns:
            col = X[feat]

            is_numeric = pd.api.types.is_numeric_dtype(col)

            if is_numeric:
                uniq = np.sort(col.dropna().unique())
                if len(uniq) <= 1:
                    continue
                thresholds = (uniq[:-1] + uniq[1:]) / 2.0

                for thr in thresholds:
                    left_mask = col <= thr
                    right_mask = ~left_mask

                    nL = int(left_mask.sum())
                    nR = int(right_mask.sum())
                    if nL < self.min_samples_leaf or nR < self.min_samples_leaf:
                        continue

                    gL = self._gini(y[left_mask])
                    gR = self._gini(y[right_mask])
                    impurity = (nL / n) * gL + (nR / n) * gR

                    if impurity < best_impurity:
                        best_impurity = impurity
                        best = {
                            "feature": feat,
                            "type": "num",
                            "threshold": float(thr),
                            "left_mask": left_mask,
                            "right_mask": right_mask,
                        }
            else:
                uniq = col.dropna().unique()
                if len(uniq) <= 1:
                    continue
                for cat in uniq:
                    left_mask = col == cat
                    right_mask = ~left_mask

                    nL = int(left_mask.sum())
                    nR = int(right_mask.sum())
                    if nL < self.min_samples_leaf or nR < self.min_samples_leaf:
                        continue

                    gL = self._gini(y[left_mask])
                    gR = self._gini(y[right_mask])
                    impurity = (nL / n) * gL + (nR / n) * gR

                    if impurity < best_impurity:
                        best_impurity = impurity
                        best = {
                            "feature": feat,
                            "type": "cat",
                            "category": cat,
                            "left_mask": left_mask,
                            "right_mask": right_mask,
                        }

        return best

    def _build_tree(self, X, y, depth):
        """
        Node format:
        - leaf: {"is_leaf": True, "pred": 0/1}
        - internal:
          {
            "is_leaf": False,
            "pred": majority_class_at_node,
            "feature": ...,
            "type": "num"/"cat",
            "threshold"/"category": ...,
            "left": node,
            "right": node
          }
        """
        node_pred = self._majority_class(y)

        if depth >= self.max_depth or len(y) < self.min_samples_split or self._gini(y) == 0.0:
            return {"is_leaf": True, "pred": node_pred}

        split = self._best_split(X, y)
        if split is None:
            return {"is_leaf": True, "pred": node_pred}

        left_mask = split["left_mask"]
        right_mask = split["right_mask"]

        if left_mask.sum() == 0 or right_mask.sum() == 0:
            return {"is_leaf": True, "pred": node_pred}

        left_node = self._build_tree(X[left_mask].reset_index(drop=True),
                                     y[left_mask].reset_index(drop=True),
                                     depth + 1)
        right_node = self._build_tree(X[right_mask].reset_index(drop=True),
                                      y[right_mask].reset_index(drop=True),
                                      depth + 1)

        node = {
            "is_leaf": False,
            "pred": node_pred,
            "feature": split["feature"],
            "type": split["type"],
            "left": left_node,
            "right": right_node,
        }
        if split["type"] == "num":
            node["threshold"] = split["threshold"]
        else:
            node["category"] = split["category"]
        return node

    def _predict_one(self, row, node):
        while not node["is_leaf"]:
            feat = node["feature"]
            if node["type"] == "num":
                thr = node["threshold"]
                go_left = row[feat] <= thr
            else:
                cat = node["category"]
                go_left = row[feat] == cat

            node = node["left"] if go_left else node["right"]

        return int(node["pred"])


In [3]:
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

def accuracy(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    return (y_true == y_pred).mean()

def confusion_matrix(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tp = np.sum((y_true == 1) & (y_pred == 1))
    return np.array([[tn, fp],
                     [fn, tp]])

def precision_recall_f1(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp = cm[0, 0], cm[0, 1]
    fn, tp = cm[1, 0], cm[1, 1]
    precision = tp / (tp + fp + 1e-12)
    recall    = tp / (tp + fn + 1e-12)
    f1        = 2 * precision * recall / (precision + recall + 1e-12)
    return precision, recall, f1

train_path = "s1_part2_dataset.csv"
test_path  = "s1_part2_test.csv"

df_train = pd.read_csv(train_path)
df_test  = pd.read_csv(test_path)

skill_map = {"Low": 0, "Medium": 1, "High": 2}
df_train["prog_skill"] = df_train["prog_skill"].map(skill_map)
df_test["prog_skill"]  = df_test["prog_skill"].map(skill_map)

features = ["group_size", "meetings_per_week", "prog_skill", "avg_previous_grade"]

X = df_train[features].copy()
y = df_train["Is_finished"].astype(int).copy()

idx = np.random.permutation(len(df_train))
split = int(0.7 * len(df_train))
tr_idx = idx[:split]
va_idx = idx[split:]

X_tr = X.iloc[tr_idx].reset_index(drop=True)
y_tr = y.iloc[tr_idx].reset_index(drop=True)
X_va = X.iloc[va_idx].reset_index(drop=True)
y_va = y.iloc[va_idx].reset_index(drop=True)

X_te = df_test[features].copy()

best_depth = None
best_acc = -1
best_model = None

for d in range(1, 11):
    model = MyDecisionTree(max_depth=d)
    model.fit(X_tr, y_tr)
    yhat = model.predict(X_va)

    acc = accuracy(y_va, yhat)
    if acc > best_acc:
        best_acc = acc
        best_depth = d
        best_model = model

print("Best depth:", best_depth)
print("Accuracy:", best_acc)
yhat_best = best_model.predict(X_va)
print("Precision, Recall, F1:", precision_recall_f1(y_va, yhat_best))
print("Confusion Matrix:\n", confusion_matrix(y_va, yhat_best))

stump = MyDecisionTree(max_depth=1)
stump.fit(X_tr, y_tr)
yhat1 = stump.predict(X_va)

print("\nDepth = 1 results")
print("Accuracy:", accuracy(y_va, yhat1))
print("Precision, Recall, F1:", precision_recall_f1(y_va, yhat1))
print("Confusion Matrix:\n", confusion_matrix(y_va, yhat1))

final_model = MyDecisionTree(max_depth=best_depth)
final_model.fit(X, y)
test_pred = final_model.predict(X_te).astype(int)

out = pd.DataFrame({"Is_finished": test_pred})
out.to_csv("s1_part2_test_predictions.csv", index=False)
print("\nSaved: s1_part2_test_predictions.csv")


Best depth: 7
Accuracy: 0.9633333333333334
Precision, Recall, F1: (np.float64(0.9704433497536898), np.float64(0.9752475247524705), np.float64(0.9728395061723347))
Confusion Matrix:
 [[ 92   6]
 [  5 197]]

Depth = 1 results
Accuracy: 0.9466666666666667
Precision, Recall, F1: (np.float64(0.9558823529411719), np.float64(0.9653465346534605), np.float64(0.9605911330044213))
Confusion Matrix:
 [[ 89   9]
 [  7 195]]

Saved: s1_part2_test_predictions.csv


In [4]:
TreeClass = None
for cand in ["MyDecisionTree", "MyDecisionTreeClassifier", "DecisionTree", "MyTree"]:
    if cand in globals():
        TreeClass = globals()[cand]
        print("Using class:", cand)
        break

if TreeClass is None:
    raise NameError(
        "No class like MyDecisionTree was found in the notebook. "
        "Please run the cell where the tree class is defined first."
    )

if "X" not in globals() or "y" not in globals():
    raise NameError(
        "Variables X and y do not exist. "
        "Please run the cell where X and y are created and split first."
    )

model_to_draw = TreeClass(max_depth=7)
model_to_draw.fit(X, y)

def node_to_str(node, indent="", branch=""):
    if node.get("is_leaf", False):
        return f"{indent}{branch}Leaf -> pred={node['pred']}\n"

    feat = node["feature"]
    if node["type"] == "num":
        rule_l = f"{feat} <= {node['threshold']:.4g}"
        rule_r = f"{feat} > {node['threshold']:.4g}"
    else:
        rule_l = f"{feat} == {node['category']}"
        rule_r = f"{feat} != {node['category']}"

    s = f"{indent}{branch}Node(pred={node['pred']}): {rule_l}\n"
    s += node_to_str(node["left"],  indent + "   ", "├─ ")
    s += f"{indent}   └─ else: {rule_r}\n"
    s += node_to_str(node["right"], indent + "      ", "└─ ")
    return s

root_node = None
if hasattr(model_to_draw, "root"):
    root_node = model_to_draw.root
elif hasattr(model_to_draw, "root_"):
    root_node = model_to_draw.root_

if root_node is None:
    raise AttributeError(
        "The model has no root attribute. "
        "If your tree structure is different, tell me so I can adapt the printer."
    )

print(node_to_str(root_node))


Using class: MyDecisionTree
Node(pred=1): avg_previous_grade <= 14.07
   ├─ Node(pred=0): avg_previous_grade <= 13.7
      ├─ Node(pred=0): avg_previous_grade <= 12.62
         ├─ Node(pred=0): avg_previous_grade <= 12.21
            ├─ Leaf -> pred=0
            └─ else: avg_previous_grade > 12.21
               └─ Node(pred=0): avg_previous_grade <= 12.24
                  ├─ Leaf -> pred=1
                  └─ else: avg_previous_grade > 12.24
                     └─ Leaf -> pred=0
         └─ else: avg_previous_grade > 12.62
            └─ Node(pred=0): avg_previous_grade <= 12.64
               ├─ Leaf -> pred=1
               └─ else: avg_previous_grade > 12.64
                  └─ Node(pred=0): group_size <= 2.5
                     ├─ Leaf -> pred=0
                     └─ else: group_size > 2.5
                        └─ Node(pred=0): meetings_per_week <= 2.5
                           ├─ Node(pred=0): group_size <= 3.5
                              ├─ Leaf -> pred=0
          

## Part 2

## 🧹 Data Preprocessing  
The preprocessing part is given to you as below.

In [26]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

In [24]:
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # Lowercase
    text = text.lower()
    # Remove punctuation and special characters (keep only a-z and spaces)
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenize (split by space)
    words = text.split()
    # Remove stopwords & Lemmatize
    clean_words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return " ".join(clean_words)

# Apply clean_text() to each email message in your dataframe

In [29]:
import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import BernoulliNB, MultinomialNB
from sklearn.tree import DecisionTreeClassifier

SEED = 42
np.random.seed(SEED)

def accuracy(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    return (y_true == y_pred).mean()

def confusion_matrix(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tp = np.sum((y_true == 1) & (y_pred == 1))
    return np.array([[tn, fp],
                     [fn, tp]])

def precision_recall_f1(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp = cm[0, 0], cm[0, 1]
    fn, tp = cm[1, 0], cm[1, 1]
    precision = tp / (tp + fp + 1e-12)
    recall    = tp / (tp + fn + 1e-12)
    f1        = 2 * precision * recall / (precision + recall + 1e-12)
    return precision, recall, f1

In [30]:
train_path = "part2_dataset.csv"
test_path  = "part2_test.csv"

df = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

possible_text_cols  = ["message", "text", "email", "content", "body"]
possible_label_cols = ["is_important", "label", "y", "target"]

text_col = None
for c in possible_text_cols:
    if c in df.columns:
        text_col = c
        break
if text_col is None:
    obj_cols = [c for c in df.columns if df[c].dtype == "object"]
    text_col = obj_cols[0]

label_col = None
for c in possible_label_cols:
    if c in df.columns:
        label_col = c
        break
if label_col is None:
    non_obj = [c for c in df.columns if df[c].dtype != "object"]
    label_col = non_obj[-1]

print("Detected text_col:", text_col)
print("Detected label_col:", label_col)

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = "" if pd.isna(text) else str(text)
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    clean_words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return " ".join(clean_words)

df["clean"] = df[text_col].apply(clean_text)
df_test["clean"] = df_test[text_col].apply(clean_text)

y = df[label_col].astype(int).reset_index(drop=True)

display(df[[text_col, "clean"]].head())


Detected text_col: message_body
Detected label_col: is_important


,message_body,clean
0,Dear students the midterm exam for AI course i...,dear student midterm exam ai course scheduled ...
1,Get 50% discount on all pepperoni pizzas at Te...,get discount pepperoni pizza tehran fastfood t...
2,Dr. Fadaei has agreed to extend the deadline f...,dr fadaei agreed extend deadline homework frid...
3,Amazing trip to the North of Iran. Villa with ...,amazing trip north iran villa beach view book ...
4,Please ensure you zip your project files befor...,please ensure zip project file final submissio...


In [ ]:
np.random.seed(SEED)
idx = np.random.permutation(len(df))
split = int(0.7 * len(df))

tr_idx = idx[:split]
va_idx = idx[split:]

X_tr_text = df.loc[tr_idx, "clean"].reset_index(drop=True)
y_tr      = y.loc[tr_idx].reset_index(drop=True)

X_va_text = df.loc[va_idx, "clean"].reset_index(drop=True)
y_va      = y.loc[va_idx].reset_index(drop=True)

X_te_text = df_test["clean"].reset_index(drop=True)

def run_setting(binary_mode):
    mode_name = "BINARY" if binary_mode else "COUNT"

    vec = CountVectorizer(binary=binary_mode)
    X_tr = vec.fit_transform(X_tr_text)
    X_va = vec.transform(X_va_text)
    X_te = vec.transform(X_te_text)

    nb = BernoulliNB() if binary_mode else MultinomialNB()

    best_dt = None
    best_depth = None
    best_acc = -1

    for d in [4, 5]:
        dt = DecisionTreeClassifier(max_depth=d, random_state=SEED)
        dt.fit(X_tr, y_tr)
        yhat_dt = dt.predict(X_va)
        acc_dt = accuracy(y_va, yhat_dt)
        if acc_dt > best_acc:
            best_acc = acc_dt
            best_dt = dt
            best_depth = d

    nb.fit(X_tr, y_tr)

    yhat_nb = nb.predict(X_va)
    yhat_dt_best = best_dt.predict(X_va)

    print("\n==============================")
    print("Vector mode", mode_name)

    print("\nNaive Bayes results")
    print("Accuracy", accuracy(y_va, yhat_nb))
    print("Precision Recall F1", precision_recall_f1(y_va, yhat_nb))
    print("Confusion Matrix\n", confusion_matrix(y_va, yhat_nb))

    print(f"\nDecision Tree results max_depth {best_depth}")
    print("Accuracy", accuracy(y_va, yhat_dt_best))
    print("Precision Recall F1", precision_recall_f1(y_va, yhat_dt_best))
    print("Confusion Matrix\n", confusion_matrix(y_va, yhat_dt_best))

    if accuracy(y_va, yhat_nb) >= accuracy(y_va, yhat_dt_best):
        best_model = nb
        best_name = "nb"
    else:
        best_model = best_dt
        best_name = f"dt_d{best_depth}"

    test_pred = best_model.predict(X_te).astype(int)
    out = pd.DataFrame({"is_important": test_pred})

    out_path = f"part2_test_predictions_{mode_name.lower()}_{best_name}.csv"
    out.to_csv(out_path, index=False)
    print("\nSaved:", out_path)

run_setting(binary_mode=True)
run_setting(binary_mode=False)


Vector mode BINARY

Naive Bayes results
Accuracy 0.8333333333333334
Precision Recall F1 (np.float64(0.78260869565214), np.float64(0.9473684210525818), np.float64(0.8571428571423209))
Confusion Matrix
 [[12  5]
 [ 1 18]]

Decision Tree results max_depth 4
Accuracy 0.7777777777777778
Precision Recall F1 (np.float64(0.7037037037036776), np.float64(0.9999999999999475), np.float64(0.8260869565212183))
Confusion Matrix
 [[ 9  8]
 [ 0 19]]

Saved: part2_test_predictions_binary_nb.csv

Vector mode COUNT

Naive Bayes results
Accuracy 0.9722222222222222
Precision Recall F1 (np.float64(0.9999999999999445), np.float64(0.9473684210525818), np.float64(0.9729729729724208))
Confusion Matrix
 [[17  0]
 [ 1 18]]

Decision Tree results max_depth 4
Accuracy 0.7777777777777778
Precision Recall F1 (np.float64(0.7037037037036776), np.float64(0.9999999999999475), np.float64(0.8260869565212183))
Confusion Matrix
 [[ 9  8]
 [ 0 19]]

Saved: part2_test_predictions_count_nb.csv


## Part 3

In [33]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

SEED = 42
np.random.seed(SEED)

df = pd.read_csv("part3_dataset.csv")
df_test = pd.read_csv("part3_test.csv")

display(df.head())


,days_before_started_study,num_slides,num_assignments_done,avg_sleep_last_week,coffee_cups_per_day,hours_studied_last_week,avg_quiz_score,midterm_score,assignments_done_ratio,class_attendance_ratio,sleep_hours_before_exam,stress_level,ai_background,mentality_type,exam_time,crisis_type
0,2,253,7,4.4,3,3.7,10.2,12.5,0.80,0.83,8.4,4.0,basic,project_heavy,morning,Denial_mode
1,4,90,6,6.8,2,36.4,17.5,15.2,0.62,0.71,4.0,3.0,none,project_heavy,morning,panic_mode
2,0,255,1,8.1,1,3.6,10.2,7.7,0.14,0.85,8.2,3.0,basic,memorization_heavy,Evening,Denial_mode
3,5,169,5,8.6,1,30.4,12.1,14.7,0.50,0.14,6.9,2.0,basic,memorization_heavy,Evening,No_issue
4,8,149,2,6.2,1,34.1,14.9,15.5,0.25,0.79,8.7,3.0,strong,project_heavy,Evening,No_issue


In [34]:
target_col = "crisis_type"

X = df.drop(columns=[target_col])
y = df[target_col]

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical:", cat_cols)
print("Numerical:", num_cols)

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols)
    ]
)

X_tr, X_va, y_tr, y_va = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y
)


Categorical: ['ai_background', 'mentality_type', 'exam_time']
Numerical: ['days_before_started_study', 'num_slides', 'num_assignments_done', 'avg_sleep_last_week', 'coffee_cups_per_day', 'hours_studied_last_week', 'avg_quiz_score', 'midterm_score', 'assignments_done_ratio', 'class_attendance_ratio', 'sleep_hours_before_exam', 'stress_level']


In [36]:
model = Pipeline(steps=[
    ("prep", preprocess),
    ("dt", DecisionTreeClassifier(max_depth=5, random_state=SEED))
])

model.fit(X_tr, y_tr)

y_pred = model.predict(X_va)

acc = accuracy_score(y_va, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(
    y_va, y_pred, average="macro"
)

print("Overall Accuracy:", acc)
print("Macro Precision:", prec)
print("Macro Recall:", rec)
print("Macro F1:", f1)
print("Confusion Matrix:\n", confusion_matrix(y_va, y_pred))


Overall Accuracy: 0.9233333333333333
Macro Precision: 0.9134193262273015
Macro Recall: 0.9261903351359053
Macro F1: 0.918212172251779
Confusion Matrix:
 [[248   9   7]
 [ 37 422  19]
 [ 12   8 438]]


In [37]:
model.fit(X, y)

test_pred = model.predict(df_test)

out = pd.DataFrame({"crisis_type": test_pred})
out.to_csv("part3_test_predictions.csv", index=False)

print("Saved: part3_test_predictions.csv")


Saved: part3_test_predictions.csv


## Part 4

In [ ]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

df = pd.read_csv("part4_dataset.csv")
df_test = pd.read_csv("part4_test.csv")

target_col = "final_exam_score"

X = df.drop(columns=[target_col])
y = df[target_col].astype(float)

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical:", cat_cols)
print("Numerical:", num_cols)

preprocess = ColumnTransformer(
    transformers=[
        ("cat", Pipeline(steps=[
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
        ("num", Pipeline(steps=[
            ("imp", SimpleImputer(strategy="median"))
        ]), num_cols),
    ],
    remainder="drop"
)

X_tr, X_va, y_tr, y_va = train_test_split(
    X, y, test_size=0.3, random_state=SEED
)

def mse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.mean((y_true - y_pred) ** 2)

def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.mean(np.abs(y_true - y_pred))

def r2_score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1.0 - ss_res / (ss_tot + 1e-12)

print("Cell 1 ready.")


Categorical: ['ai_background', 'mentality_type', 'exam_time', 'crisis_type']
Numerical: ['days_before_started_study', 'num_slides', 'num_assignments_done', 'avg_sleep_last_week', 'coffee_cups_per_day', 'hours_studied_last_week', 'avg_quiz_score', 'midterm_score', 'assignments_done_ratio', 'class_attendance_ratio', 'sleep_hours_before_exam', 'stress_level']
Cell 1 ready.


In [ ]:

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline


models = {
    "DecisionTreeRegressor": DecisionTreeRegressor(max_depth=6, random_state=SEED),
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        random_state=SEED,
        n_jobs=-1
    ),
}

results = {}

for name, reg in models.items():
    pipe = Pipeline(steps=[
        ("prep", preprocess),  
        ("model", reg)
    ])

    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_va)

    results[name] = {
        "MSE": mse(y_va, pred),
        "MAE": mae(y_va, pred),
        "R2":  r2_score(y_va, pred),
        "model": pipe
    }

    print("\n" + "="*30)
    print("Model:", name)
    print("MSE:", results[name]["MSE"])
    print("MAE:", results[name]["MAE"])
    print("R2 :", results[name]["R2"])

best_name = min(results.keys(), key=lambda k: results[k]["MSE"])
print("\nBest by MSE on validation:", best_name)
best_model = results[best_name]["model"]



Model: DecisionTreeRegressor
MSE: 6.820529459837005
MAE: 1.8829026441874035
R2 : 0.653245510791466

Model: RandomForestRegressor
MSE: 6.182640224537039
MAE: 1.735619166666667
R2 : 0.6856756846160283

Best by MSE on validation: RandomForestRegressor


In [ ]:

best_model = results[best_name]["model"]

best_model.fit(X, y)

test_pred = best_model.predict(df_test).astype(float)

out = pd.DataFrame({"final_exam_score": test_pred})
out.to_csv(f"part4_test_predictions_{best_name}.csv", index=False)

print("Saved", f"part4_test_predictions_{best_name}.csv")


Saved part4_test_predictions_RandomForestRegressor.csv


In [ ]:

import numpy as np
import pandas as pd


rng = np.random.default_rng(SEED)

i_star = 0
x_star = X_va.iloc[[i_star]]
y_star = float(y_va.iloc[i_star])
print("y_star", y_star)

T = 100
sub_frac = 0.30

preds_dt = []
preds_rf = []

for t in range(T):
    n_sub = int(len(X_tr) * sub_frac)
    idx = rng.choice(len(X_tr), size=n_sub, replace=False)

    X_sub = X_tr.iloc[idx]
    y_sub = y_tr.iloc[idx]

    dt_pipe = Pipeline(steps=[
        ("prep", preprocess),
        ("reg", DecisionTreeRegressor(max_depth=6, random_state=SEED + t))
    ])

    rf_pipe = Pipeline(steps=[
        ("prep", preprocess),
        ("reg", RandomForestRegressor(n_estimators=300, random_state=SEED + t, n_jobs=-1))
    ])

    dt_pipe.fit(X_sub, y_sub)
    rf_pipe.fit(X_sub, y_sub)

    preds_dt.append(float(dt_pipe.predict(x_star)[0]))
    preds_rf.append(float(rf_pipe.predict(x_star)[0]))

def bias_var(preds, y_star):
    preds = np.array(preds, dtype=float)
    mean_pred = float(np.mean(preds))
    bias2 = float((mean_pred - y_star) ** 2)
    var = float(np.var(preds))
    return mean_pred, bias2, var, float(bias2 + var)

mean_dt, bias2_dt, var_dt, total_dt = bias_var(preds_dt, y_star)
mean_rf, bias2_rf, var_rf, total_rf = bias_var(preds_rf, y_star)

print("\nDecisionTree")
print("mean_pred", mean_dt)
print("bias2", bias2_dt)
print("var", var_dt)
print("bias2+var", total_dt)

print("\nRandomForest")
print("mean_pred", mean_rf)
print("bias2", bias2_rf)
print("var", var_rf)
print("bias2+var", total_rf)


y_star 12.4

DecisionTree
mean_pred 12.467160151925539
bias2 0.00451048600666138
var 3.0655863856911885
bias2+var 3.07009687169785

RandomForest
mean_pred 12.208266666666669
bias2 0.036761671111110505
var 0.4595776177777768
bias2+var 0.4963392888888873
